# LF1–5 — correctness vs ground-truth

Sinh nhãn **độ hữu dụng** theo hướng *task-driven*: một ảnh phù hợp cho tác vụ khi
mô hình hạ nguồn của tác vụ đó xử lý **đúng** so với ground-truth. Tín hiệu mạnh nhất
trong khung weak-supervision (xem `AGENTS.md`, `docs/Labeling_Plan.md`).

Bộ 5 nhãn (phân theo *required-view* — ảnh phải thể hiện đúng đối tượng/góc nhìn):
- `1_maturity_evaluation` — độ chín; GT = mức `dry/green/tender` (Roboflow, YOLO .txt)
- `2_foliar_disease` — bệnh lá; GT = Gray Leaf Spot, Leaf Rot
- `3_trunk_disease` — bệnh thân; GT = Stem Bleeding
- `4_crown_disease` — bệnh đọt/crown; GT = Bud Rot
- `5_petiole` — tình trạng tàu lá qua cuống lá/độ rủ (Bud Root Dropping); GT = Bud Root Dropping

Mỗi ảnh chỉ có GT cho **đúng một tác vụ** (theo nguồn của nó). Mỗi tác vụ ghi ra **một file riêng**
`labels/votes/lf<N>_<tên>.csv` (schema chung `src/utils/lf_io.ipynb`).
Mô hình hạ nguồn là **hàm giữ chỗ** (placeholder) để chạy được trước khi tích hợp.

> LF1 (độ chín) có notebook chuyên dụng `lf1_maturity_yolov8.ipynb` (bản thật). Ở notebook này LF1 chỉ là placeholder.

In [ ]:
# Cấu hình
from pathlib import Path
import csv
import numpy as np

ROOT = Path.cwd()
if not (ROOT/'Dataset').exists():
    for p in ROOT.parents:
        if (p/'Dataset').exists(): ROOT = p; break

TASKS = ["1_maturity_evaluation", "2_foliar_disease", "3_trunk_disease",
         "4_crown_disease", "5_petiole"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Ánh xạ thư mục bệnh -> tác vụ (mỗi bệnh một required-view riêng)
DISEASE_TASK = {
    "Gray Leaf Spot": "2_foliar_disease", "Leaf Rot": "2_foliar_disease",
    "Stem Bleeding": "3_trunk_disease",
    "Bud Rot": "4_crown_disease",
    "Bud Root Dropping": "5_petiole",
}
# data.yaml Roboflow: names=['dry','green','tender'] -> class 0/1/2 = mức độ chín
YOLO_STAGE = {0: "dry", 1: "green", 2: "tender"}
# Nap module dung chung (schema phieu + writer). Khong import duoc .ipynb -> dung %run.
UTILS = ROOT/"src"/"utils"/"lf_io.ipynb"
if not UTILS.exists():
    raise SystemExit("Không thấy " + str(UTILS))
get_ipython().run_line_magic("run", str(UTILS))
print("ROOT =", ROOT)

## 1. Đọc ground-truth

In [ ]:
def read_yolo_stages(label_txt):
    """Mức độ chín (dry/green/tender) của các trái trong ảnh, theo class YOLO."""
    if not label_txt.exists(): return []
    out = []
    for line in label_txt.read_text().splitlines():
        if line.strip(): out.append(YOLO_STAGE.get(int(line.split()[0]), "?"))
    return out

def iter_roboflow(root):
    base = root/"Dataset"/"coconut-veirf-v5"
    for split in ("train", "valid", "test"):
        img_dir, lbl_dir = base/split/"images", base/split/"labels"
        if not img_dir.exists(): continue
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() in IMG_EXTS:
                yield img, split, read_yolo_stages(lbl_dir/(img.stem + ".txt"))

def iter_disease(root):
    base = root/"Dataset"/"Coconut Tree Disease Dataset"
    if not base.exists(): return
    for folder in sorted(base.iterdir()):
        if not folder.is_dir(): continue
        gt_task = DISEASE_TASK.get(folder.name)
        for img in sorted(folder.rglob("*")):
            if img.suffix.lower() in IMG_EXTS:
                yield img, folder.name, gt_task

## 2. Mô hình hạ nguồn (hàm giữ chỗ — thay bằng mô hình thực)

Mặc định trả `None` → ảnh "chưa đánh giá được" (cột nhãn để trống), để notebook chạy được
trước khi có mô hình thực. Mỗi tác vụ bệnh có một bộ phân loại riêng trả về True/False
(ảnh có đúng là lớp bệnh đó / thể hiện đủ để chẩn).

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class DownstreamModels:
    predict_maturity: Callable = lambda p: None    # -> 'dry'/'green'/'tender'
    predict_foliar: Callable = lambda p: None      # -> bool (bệnh lá)
    predict_trunk: Callable = lambda p: None       # -> bool (bệnh thân)
    predict_crown: Callable = lambda p: None       # -> bool (bệnh đọt/crown)
    predict_petiole: Callable = lambda p: None # -> bool (tình trạng tàu lá)

## 3. Correctness -> nhãn hữu dụng

In [ ]:
def maturity_correct(pred_stage, gt_stages):
    """Đúng nếu mức chín dự đoán nằm trong tập mức thật của ảnh."""
    if pred_stage is None or not gt_stages: return None
    return int(pred_stage in gt_stages)

def cls_correct(pred_is_class):
    """Ảnh (thuộc đúng tác vụ này) dùng được nếu model chẩn đúng. None = chưa có model."""
    if pred_is_class is None: return None
    return int(bool(pred_is_class))

## 4. Chạy & ghi mỗi tác vụ một file `labels/votes/lf<N>_<tên>.csv`

In [ ]:
LF_OF_TASK = {
    "1_maturity_evaluation": ("lf1_maturity", "lf1_maturity.csv"),
    "2_foliar_disease":      ("lf2_foliar",   "lf2_foliar.csv"),
    "3_trunk_disease":       ("lf3_trunk",    "lf3_trunk.csv"),
    "4_crown_disease":       ("lf4_crown",    "lf4_crown.csv"),
    "5_petiole":             ("lf5_petiole",  "lf5_petiole.csv"),
}


def build(root, models, votes_dir):
    per_task = {}
    for t in TASKS:
        per_task[t] = []
    # Roboflow -> do chin
    for img, split, gt_stages in iter_roboflow(root):
        vote = maturity_correct(models.predict_maturity(img), gt_stages)
        row = make_vote(
            lf=LF_OF_TASK["1_maturity_evaluation"][0],
            image_id=img.stem,
            task="1_maturity_evaluation",
            vote=vote,
            source="coconut-veirf-v5/" + split,
            path=str(img.relative_to(root)),
        )
        if row is not None:
            per_task["1_maturity_evaluation"].append(row)
    # Benh -> moi anh gan dung tac vu goc (noi co GT)
    disease_pred = {
        "2_foliar_disease": models.predict_foliar,
        "3_trunk_disease": models.predict_trunk,
        "4_crown_disease": models.predict_crown,
        "5_petiole": models.predict_petiole,
    }
    for img, folder, gt_task in iter_disease(root):
        if gt_task not in disease_pred:
            continue
        vote = cls_correct(disease_pred[gt_task](img))
        row = make_vote(
            lf=LF_OF_TASK[gt_task][0],
            image_id=img.stem,
            task=gt_task,
            vote=vote,
            source="disease/" + folder,
            path=str(img.relative_to(root)),
        )
        if row is not None:
            per_task[gt_task].append(row)
    votes_dir.mkdir(parents=True, exist_ok=True)
    for t in TASKS:
        fname = LF_OF_TASK[t][1]
        write_lf_votes(votes_dir / fname, per_task[t])

### Tích hợp mô hình thực rồi chạy

```python
models = DownstreamModels(
    predict_maturity  = lambda p: my_maturity_clf.predict(p),   # 'dry'/'green'/'tender'
    predict_foliar    = lambda p: my_foliar_clf.predict(p),     # bool
    predict_trunk     = lambda p: my_trunk_clf.predict(p),      # bool
    predict_crown     = lambda p: my_crown_clf.predict(p),      # bool
    predict_petiole = lambda p: my_petiole_clf.predict(p),  # bool
)
```

In [ ]:
models = DownstreamModels()      # hàm giữ chỗ: chưa tích hợp mô hình -> cột nhãn để trống
build(ROOT, models, ROOT/"labels"/"votes")